# GRU — is the un-edited part of the latent the *past*?

**Hypothesis under test (Sevan, 2026-08-13).** *The reason our edits are failing is because the extra
information in the latent world state, which we're not editing (outside the probe's row space), is
information pertaining to the previous frames / history.*

This is the first hypothesis in the thread that names a **specific content** for the un-edited
complement. Every prior negative describes that complement only geometrically (at/below chance in the
probe row space; ~35–42% of `h` not a function of `(pos, vel)`; orthogonal to the true edit direction).
"It is the past" is a testable answer, and this notebook tests it in three steps:

1. **§1 — Is the past readable from `h`?** Probe `h_t → pos(t−k)` for k = 0…20, against the null that
   says nothing about the past is stored.
2. **§2 — Is the un-edited complement the past?** Decompose the fiber residual (the part of `h` that is
   *not* a function of the present `(pos, vel)`) and ask what predicts it.
3. **§3–§5 — Does writing the whole history fix the edit?** Translate the model's believed history
   rigidly by the teleport displacement, through the latent, and compare against **the identical content
   delivered through the observation channel**.

> **Design note — why the §3 comparison is the point.** Every editor in this thread that *works*
> (counterfactual overwrite +0.68, DiT counterfactual window write +0.71, transformer history overwrite
> +0.63, freeze-time +0.52) supplies a velocity-consistent *translated history* through **observations**;
> every editor that fails writes one frame's position into the **latent**. §3 holds the content, the
> displacement and the number of history frames fixed and varies **only the channel**. Whatever the
> outcome, it is interpretable — see the pre-registered table in §3.

**Direction brief:** `research/directions/history-editing.md` · **Companion notebook:**
`transformer_history_editing.ipynb` (same hypothesis where the history *is* the carried state).

In [ ]:
# [1] setup
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

ROOT = Path("/home/sevan/research/physically-implicit-modeling")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "notebooks/experiments/editability/history_editing"))

from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio
from history_tools import (
    chance_fraction,
    effective_rank,
    lag_probe_curve,
    ray_centroid,
    subspace_fraction,
    waterfall_grid,
)
from pim.extractors import fit_readability_probes
from pim.figures.theme import style_ax
from pim.simulator.renderer import render_frame
from pim.simulator.sim import SimConfig
from pim.world_models import load_checkpoint, load_dataset

DEVICE = "cuda"
N_OBJ = 2              # objects in the world
K = 15                 # rollout steps scored after the edit
N_PROBE = 1500         # test sequences used for the §1/§2 probes
N_EDIT = 256           # held-out edit samples scored in §3
N_BANK = 2000          # edits sequences used to fit the editing probes
TMIN_PROBE = 20        # §1/§2 use frames t >= 20 ("late-t": the belief has converged, lag 20 available)
TMIN_EDIT = 16         # §3 probe rows: frames 16..19 — pre-edit, and IDENTICAL for every n
NS = [0, 1, 2, 4, 8, 16]   # history depths swept in §3 (n = number of PAST frames written, beyond the current one)
LAGS = list(range(0, 21))
MLP_LAGS = [0, 2, 5, 10, 15, 20]

# Okabe-Ito (colour-blind safe) — light academic theme for every metrics figure in this notebook
OI = {
    "blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
    "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#5a5a5a",
}
torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| device", DEVICE, "|", torch.cuda.get_device_name(0))

## Definitions

Every non-obvious term and every metric used below, with its formula, units and better-direction.
§4-editability metric names/formulas are copied verbatim from the canonical registry
`../METRICS_AND_EDITORS.md` and are computed by `scripts/editability_metrics.py` — none are re-derived here.

### Run / data provenance

| name | what it is |
|---|---|
| **GRU · 256 hidden · dataset 4** (`runs/controls/H256`) | The thread's reference GRU: 256-unit GRU, pure next-step-prediction MSE, 400 epochs, seed 0, trained on `datasets/4_fixed_refl_inview`. This is the checkpoint `editor_gallery`, `iterative_probing` and `metric_corrected_edits` all use, so §3 numbers are directly comparable to their published values. Registry row: `../controls/CONTROL_RUNS.md`. |
| **dataset 4** (`datasets/4_fixed_refl_inview`) | 2 objects, 40 frames, `obs_res = 128` (a 1-D perspective scan), open boundary, fixed reflectivities, always-in-frustum, radius 0.5, **`obs_noise_std = 0.2`** (sensing noise) and **`position_noise_std = 0.04`** (world noise), `speed_noise_std = direction_noise_std = 0`. |
| **`edits` split** | Held-out sequences in which one object is **teleported** at frame `ef = 20`, preserving its velocity. Seeds 110000+, disjoint from train. |

### Terms

| term | meaning |
|---|---|
| `h_t` | the GRU hidden state after consuming observation `t`; **aligned so `h_t` encodes frame `t`**. |
| **lag `k`** | how far into the past a probe target reaches: the probe reads `h_t` and predicts `pos(t−k)`. |
| **`n`** (history depth) | in §3, the number of **past** frames written in addition to the current one, so an arm writes lags `0…n` — i.e. `n+1` frames of history. |
| **δ** (teleport displacement) | `δ = tgt_pos − (pre_pos + dt·v)` for the edited object, `0` for the other: the target minus where the object *would* have been at `ef` had it not been teleported. Adding the same `δ` at every lag translates the whole past **rigidly**, which preserves velocity — the fixed-velocity assumption. |
| **late-t** | frames `t ≥ 20`, used for §1/§2 (the belief has converged, and lag 20 is available). |
| **ballistic** | the world's motion law between edits: velocity is exactly constant (`speed_noise_std = direction_noise_std = 0`), so `pos(t−k) = pos(t) − k·dt·v(t)` **up to the per-step position noise** of 0.04. |

### §1–§2 metrics

| name | formula | units | better | notes |
|---|---|---|---|---|
| **position / velocity R²** | `1 − ‖Y − probe(h)‖²/‖Y − Ȳ‖²`, scored on **held-out sequences** against the **train** mean | — | ↑ | linear = lstsq, MLP = 2×256 ReLU; both from `pim.extractors.fit_readability_probes`, both fit on the same 80% of *sequences*. Split by sequence, never by row. |
| **direct lag R²** | the above with `Y = pos(t−k)` | — | ↑ | "how well can the past be read off `h`?" |
| **ballistic-from-`h` R²** (the null) | probe `h → (pos_t, v_t)`, then `p̂(t−k) = p̂os_t − k·dt·v̂_t` | — | ↑ | **the no-stored-history model**: nothing about the past is read. `direct − ballistic` is the load-bearing quantity — readable information about the past that is *not* already implied by the present. |
| **learned-from-readout R²** (the tighter null) | fit a free 2×256 MLP from `h`'s own `(p̂os, v̂)` readout to `pos(t−k)` | — | ↑ | removes the objection that the gap is just extrapolation inefficiency in the imposed `p − k·dt·v` formula. |
| **true-`(pos,v)` ceiling** | the same extrapolation from the **true** present `(pos_t, v_t)` | — | ↑ | the most any history-free model could score, given the world's position noise. |
| **shuffled floor** | targets permuted across sequences | — | — | the 0 line. |
| **fiber residual** | `‖h − g(pos,vel)‖ / ‖h‖`, `g` linear or MLP | frac of ‖h‖ | ↓ (0 = fully canonical) | **the "extra information" the hypothesis is about**: the part of `h` that is not a function of the present physical state. |
| **residual explained** | held-out `R² = 1 − ‖res − f(Z)‖²/‖res‖²` of a linear map from predictor set `Z` to the fiber residual | — | ↑ | every `Z` is first **residualised against the present `(pos, vel)`**, so only *beyond-present* information can score. Reported beside a **predictor-count-matched shuffled control** — with up to 1280 predictors, dimension counting alone explains a non-trivial amount and must be subtracted. |

### §3–§5 metrics — the canonical §4 set (`../METRICS_AND_EDITORS.md`)

Two ground-truth worlds are rendered at the edit frame: **`gt_edited`** (the teleport happened) and
**`gt_unedited`** (the counterfactual where it did not — the object continued along its own velocity).
Ray zones follow from them: **target** rays = where the edited object is in `gt_edited`; **ghost** rays =
where it was and now vacates; **collateral** rays = the *other* object; **differing** rays = where the two
worlds differ at all.

| name | formula | units | better | notes |
|---|---|---|---|---|
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_· = RMSE(edited₀, gt_·)` over **differing** rays; per sample then averaged | −1…+1 | ↑ | **+1** = it *is* the edited world · **0** = equidistant (ambiguous **or garbage**) · **−1** = it *is* the unedited world. Read against **that model's own unsteered row**, not against −1. |
| **Target / Ghost / Collateral RMSE** | `RMSE(edited₀, gt_edited)` over the named zone, at rollout step 0 | obs intensity | ↓ | did it appear where it should / leave where it was / leave the other object alone? |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` over the K-step rollout | obs intensity | ↓ | did the edit achieve *and hold* the true post-edit world? |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ | **> 1 = the edited rollout ended FURTHER from the truth than doing nothing** — degradation, not steering. Reported beside every success claim. |
| **‖Δh‖/‖h‖** | mean write size relative to the state norm | ratio | — | not a quality metric; it is how the **matched-norm control** is constructed and how a "gain" bought purely by write size is exposed. |
| **effective rank** | # singular values of the stacked probe above 1% of the largest | dims | — | the rank that actually carries a write. A *numeric* rank (1e-6) counts directions whose pseudoinverse gain is ~10⁶ and which no real edit can use. |
| **row-space fraction / enrichment** | `‖P_row Δh_true‖/‖Δh_true‖`, and that divided by the chance level `√(rank/H)` | ratio | ↑ | the **hard ceiling** on any injection-style editor. Chance moves with rank, so the **enrichment** is plotted, never the raw fraction. |

### Editors

| editor | mechanism |
|---|---|
| **Unsteered** | no edit; teacher-force `obs[0…ef−1]`, then free-run. The reference row. |
| **Pseudoinverse Injection (published convention)** | the thread's standard readout injection: min-norm write to `h` so a linear position probe reads `positions[ef]`. Included to tie this notebook's numbers to published ones. |
| **Latent history translation · n** | min-norm write to `h` so a **stacked** probe reads `pos(ef−1−k) + δ` at every lag `k = 0…n`. This is the hypothesis's editor: rewrite the believed *history*, rigidly translated. |
| **Latent, inconsistent history · n=8** | the same stacked probe, but `δ` applied at lag 0 **only** — the past left where it was. Isolates *consistent translated history* from *larger write subspace*. |
| **Random direction, matched norm** | a random `Δh` with the same norm as the `n=8` arm. Isolates the effect of the write's **direction** from its **size**. |
| **Observation history overwrite · n** | the identical content through the **observation channel**: teacher-force the model on `n+1` *rendered* frames of the translated world (frames `ef−1−n … ef−1`), then free-run. At large `n` this is the thread's counterfactual-overwrite oracle. |

In [ ]:
# [2] load the model and the data, and pin the provenance
model, info = load_checkpoint(ROOT / "runs/controls/H256/best_model.pt", device=DEVICE)
model.eval()
H = model.hidden_size

bundle = load_dataset(ROOT / "datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
sim = test.config["dataset"]["sim"]
dt, EF, R = float(sim["dt"]), int(edits.edit_frame), int(sim["obs_res"])

display(Markdown(f"""
| provenance | value |
|---|---|
| model | GRU · 256 hidden · dataset 4 (`runs/controls/H256/best_model.pt`) |
| hidden size `H` | {H} |
| dataset | `datasets/4_fixed_refl_inview` |
| observation | 1-D perspective scan, `obs_res` = {R} rays |
| sensing noise `obs_noise_std` | {sim['obs_noise_std']} |
| world noise `position_noise_std` | {sim['position_noise_std']} |
| speed / direction noise | {sim['speed_noise_std']} / {sim['direction_noise_std']} (velocity is **exactly** constant) |
| `dt` | {dt} |
| edit frame `ef` | {EF} |
| probe sequences (§1/§2, `test`) | {N_PROBE} |
| edit samples (§3, `edits`) | {N_EDIT} |
"""))


@torch.no_grad()
def hidden_states(obs, batch=512):
    """Per-frame flat state; index t is aligned to frame t (`h_t` encodes frame t)."""
    out = []
    for i in range(0, len(obs), batch):
        o = torch.from_numpy(obs[i : i + batch]).float().to(DEVICE)
        out.append(model.get_hidden_states(o).cpu().numpy())
    return np.concatenate(out)


@torch.no_grad()
def warm_up_to(obs, frame):
    """Teacher-force obs[0 .. frame-1]; the resulting state encodes frame `frame-1`,
    so an ordinary rollout's step 0 decodes frame `frame`."""
    o = torch.from_numpy(obs).float().to(DEVICE)
    state = None
    for t in range(frame):
        _, state = model.step(o[:, t], state)
    return model.flat_state(state)


@torch.no_grad()
def teacher_force(h_flat, frames):
    """Continue teacher-forcing an existing state on `frames` (N, n, R)."""
    state = model.state_from_flat(h_flat)
    f = torch.from_numpy(np.ascontiguousarray(frames)).float().to(DEVICE)
    for t in range(f.shape[1]):
        _, state = model.step(f[:, t], state)
    return model.flat_state(state)


@torch.no_grad()
def rollout(h_flat, steps=K):
    """Free-run from a state; step 0 decodes the edit frame (no teacher forcing)."""
    state = model.state_from_flat(torch.as_tensor(np.asarray(h_flat)).float().to(DEVICE))
    out = [model.decode(state)]
    for _ in range(steps - 1):
        p, state = model.predict_step(state)
        out.append(p)
    return torch.stack(out, 1).cpu().numpy()


def lstsq_fit(X, Y):
    return np.linalg.lstsq(np.c_[X, np.ones(len(X))], Y, rcond=None)[0]


def lstsq_apply(A, X):
    return np.c_[X, np.ones(len(X))] @ A

## §1 — Is the past readable from `h`?

The hypothesis needs the past to be *in* `h`. But a flat "past positions are decodable" curve would be
**vacuous on its own**, because this world's velocity is exactly constant: `pos(t−k) = pos(t) − k·dt·v(t)`
up to the per-step position noise. A model that stored only `(pos, vel)` would let a probe reconstruct the
past perfectly well without storing a single past frame.

So every lag is scored against **three references**:

* **ballistic-from-`h`** — the no-stored-history null. Read `(pos_t, v_t)` off `h`, then extrapolate backwards.
* **learned-from-readout** — the same null with the extrapolation *learned* rather than imposed, so a gap
  cannot be blamed on the `p − k·dt·v` formula compounding velocity error.
* **true-`(pos, v)` ceiling** — extrapolating from the *true* present state; the best any history-free model
  could do given the world's position noise.

**`direct − null` is the load-bearing quantity.**

In [ ]:
# [3] states and targets for the §1/§2 probes (test split)
Ht = hidden_states(test.obs[:N_PROBE])
T = Ht.shape[1]
pos_t = test.positions[:N_PROBE, :, :N_OBJ, :].reshape(N_PROBE, -1, N_OBJ * 2)
with h5py.File(test.h5_path, "r") as f:
    vel_t = f["velocities"][:N_PROBE, :, :N_OBJ, :].reshape(N_PROBE, -1, N_OBJ * 2)

N_TR = int(0.8 * N_PROBE)          # the sequence split used by every §1/§2 fit
Xs = Ht[:, TMIN_PROBE:T]           # (N, t, H) states at late-t
pv = np.concatenate([pos_t[:, TMIN_PROBE:T], vel_t[:, TMIN_PROBE:T]], -1)
Xtr = Xs[:N_TR].reshape(-1, H).astype(np.float64)
Xte = Xs[N_TR:].reshape(-1, H).astype(np.float64)
PVtr = pv[:N_TR].reshape(-1, 8).astype(np.float64)
PVte = pv[N_TR:].reshape(-1, 8).astype(np.float64)
print(f"states {Ht.shape} | late-t frames {TMIN_PROBE}..{T-1} | "
      f"{len(Xtr)} train rows / {len(Xte)} held-out rows (split by SEQUENCE)")

# sanity: the world really is ballistic in velocity, and only position noise breaks the identity
with h5py.File(test.h5_path, "r") as f:
    P64 = f["positions"][:64, :, :N_OBJ, :].astype(np.float64)
    V64 = f["velocities"][:64, :, :N_OBJ, :].astype(np.float64)
print(f"max |v(t+1) − v(t)| = {np.abs(V64[:, 1:] - V64[:, :-1]).max():.2e}  (velocity is exactly constant)")
print(f"max |pos(t+1) − pos(t) − dt·v(t)| = {np.abs(P64[:, :-1] + dt * V64[:, :-1] - P64[:, 1:]).max():.4f}"
      f"  (= the per-step position noise, std {sim['position_noise_std']})")

In [ ]:
# [4] lag readability — linear (every lag) and MLP (a subset), each against its nulls
curve = lag_probe_curve(Ht, pos_t, vel_t, lags=LAGS, dt=dt, t_min=TMIN_PROBE)
print(f"linear (pos,vel) readout from h:  pos R² {curve['posvel_r2']['pos']:.4f}   "
      f"vel R² {curve['posvel_r2']['vel']:.4f}")

# the MLP arm: the standard 2x256 probe, plus BOTH nulls built from the model's own MLP readout
res_pv = fit_readability_probes(Xs, pv, device=DEVICE)
with torch.no_grad():
    pv_hat_tr = res_pv["mlp"](torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    pv_hat_te = res_pv["mlp"](torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).cpu().numpy()
print(f"MLP    (pos,vel) readout from h:  R² {res_pv['mlp_r2']:.4f}  ({res_pv['spec']})")

mlp_curve = {}
for k in MLP_LAGS:
    Y = pos_t[:, TMIN_PROBE - k : T - k]
    Ytr, Yte = Y[:N_TR].reshape(-1, 4), Y[N_TR:].reshape(-1, 4)
    mu = Ytr.mean(0)
    r2 = lambda p: float(1 - ((p - Yte) ** 2).sum() / ((Yte - mu) ** 2).sum())  # noqa: E731
    direct = fit_readability_probes(Xs, Y, device=DEVICE)
    learned = fit_readability_probes(
        pv_hat_tr.reshape(N_TR, -1, 8).astype(np.float32),
        Ytr.reshape(N_TR, -1, 4).astype(np.float32),
        device=DEVICE, holdout=0.25,
    )
    with torch.no_grad():
        p_learned = learned["mlp"](torch.tensor(pv_hat_te, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    mlp_curve[k] = dict(
        direct=direct["mlp_r2"],
        imposed=r2(pv_hat_te[:, :4] - k * dt * pv_hat_te[:, 4:]),
        learned=r2(p_learned),
        ceiling=curve["ceiling"][k],
    )

rows = ["| lag k | linear direct | linear ballistic-from-h | linear gain | MLP direct | MLP ballistic (imposed) "
        "| MLP ballistic (learned) | MLP gain vs learned | true-(pos,v) ceiling | shuffled floor |",
        "|---|---|---|---|---|---|---|---|---|---|"]
for k in MLP_LAGS:
    m = mlp_curve[k]
    rows.append(
        f"| {k} | {curve['direct'][k]:.4f} | {curve['ballistic'][k]:.4f} | "
        f"{curve['direct'][k] - curve['ballistic'][k]:+.4f} | **{m['direct']:.4f}** | {m['imposed']:.4f} | "
        f"{m['learned']:.4f} | **{m['direct'] - m['learned']:+.4f}** | {m['ceiling']:.4f} | "
        f"{curve['shuffled'][k]:+.4f} |"
    )
display(Markdown("**Table 1 — position readability at lag k, against the no-stored-history nulls.** "
                 "Held-out R², sequence split, late-t frames.\n\n" + "\n".join(rows)))

In [ ]:
# [5] Fig 1 — how far back is position readable, and does h beat a model that stores nothing?
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.2))
for ax in axes:
    style_ax(ax)

ax = axes[0]
ax.plot(LAGS, [curve["direct"][k] for k in LAGS], "o-", color=OI["blue"], lw=2, ms=4,
        label="direct linear probe  h → pos(t−k)")
ax.plot(LAGS, [curve["ballistic"][k] for k in LAGS], "s--", color=OI["orange"], lw=2, ms=4,
        label="ballistic-from-h (no stored history)")
ax.plot(LAGS, [curve["ceiling"][k] for k in LAGS], ":", color=OI["grey"], lw=2,
        label="true-(pos,v) ceiling")
ax.axhline(0, color=OI["red"], lw=1, ls=":", label="shuffled floor")
ax.set_xlabel("lag k (frames into the past)")
ax.set_ylabel("held-out R²")
ax.set_title("(a) linear probe — the two curves coincide", fontsize=10)
ax.legend(fontsize=7.5, loc="lower left")
ax.set_ylim(-0.05, 1.05)

ax = axes[1]
ax.plot(LAGS, [curve["direct"][k] - curve["ballistic"][k] for k in LAGS], "o-",
        color=OI["blue"], lw=2, ms=4, label="linear")
ax.plot(MLP_LAGS, [mlp_curve[k]["direct"] - mlp_curve[k]["learned"] for k in MLP_LAGS], "D-",
        color=OI["purple"], lw=2, ms=5, label="MLP (vs learned null)")
ax.axhline(0, color=OI["grey"], lw=1, ls="--")
ax.set_xlabel("lag k (frames into the past)")
ax.set_ylabel("R² gain over the no-stored-history null")
ax.set_title("(b) information about the past NOT implied by the present", fontsize=10)
ax.legend(fontsize=8, loc="upper left")

ax = axes[2]
ax.plot(MLP_LAGS, [mlp_curve[k]["direct"] for k in MLP_LAGS], "D-", color=OI["purple"], lw=2, ms=5,
        label="direct MLP probe  h → pos(t−k)")
ax.plot(MLP_LAGS, [mlp_curve[k]["learned"] for k in MLP_LAGS], "s--", color=OI["orange"], lw=2, ms=5,
        label="MLP ballistic (learned from h's own readout)")
ax.plot(MLP_LAGS, [mlp_curve[k]["imposed"] for k in MLP_LAGS], "^:", color=OI["sky"], lw=1.6, ms=5,
        label="MLP ballistic (imposed p − k·dt·v)")
ax.plot(MLP_LAGS, [mlp_curve[k]["ceiling"] for k in MLP_LAGS], ":", color=OI["grey"], lw=2,
        label="true-(pos,v) ceiling")
ax.set_xlabel("lag k (frames into the past)")
ax.set_ylabel("held-out R²")
ax.set_title("(c) MLP probe — direct reading pulls away with lag", fontsize=10)
ax.legend(fontsize=7.5, loc="lower left")
ax.set_ylim(0.6, 1.02)

fig.suptitle("Fig 1 — position readability from h at lag k, against the model that stores no history "
             "(GRU · 256 hidden · dataset 4, held-out sequences)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

## §2 — Is the un-edited complement the past?

§1 asks whether the past is *readable*. This section asks the hypothesis directly: the complement we never
edit is, by construction, the part of `h` that is **not** a function of the present `(pos, vel)` — the
**fiber residual**. If that complement is "information pertaining to the previous frames", then past frames
should predict it.

Two candidate predictor families, both **residualised against the present `(pos, vel)` first**, so only
*beyond-present* information can score:

* **past positions** — the hypothesis in its literal form ("where the objects were").
* **past observations** — the raw sensor frames. These carry information `(pos, vel)` cannot: noise
  realisations, silhouette detail, occlusion structure.

Every regression is reported beside a **predictor-count-matched shuffled control** (rows permuted across
sequences), because with up to 1280 predictors, dimension counting alone explains a non-trivial fraction.

In [ ]:
# [6] the fiber residual, and what explains it
G_lin = lstsq_fit(PVtr, Xtr)                       # linear g: (pos,vel) -> h
res_tr = Xtr - lstsq_apply(G_lin, PVtr)
res_te = Xte - lstsq_apply(G_lin, PVte)
fiber_lin = float(np.linalg.norm(res_te) / np.linalg.norm(Xte))

r_fib = fit_readability_probes(pv, Xs, device=DEVICE)   # MLP g: (pos,vel) -> h
with torch.no_grad():
    h_hat_te = r_fib["mlp"](torch.tensor(PVte, dtype=torch.float32, device=DEVICE)).cpu().numpy()
fiber_mlp = float(np.linalg.norm(Xte - h_hat_te) / np.linalg.norm(Xte))
print(f"fiber residual ‖h − g(pos,vel)‖/‖h‖:   linear {fiber_lin:.4f}   MLP {fiber_mlp:.4f}")
print("This is the 'extra information' the hypothesis is about — what predicts it?\n")

rng = np.random.default_rng(0)


def explain_residual(source, frame_offsets, label):
    """Held-out R² of a linear map from `source` at the given past offsets to the fiber residual,
    after removing everything the PRESENT (pos, vel) already implies. Plus a shuffled control."""
    Ztr = np.concatenate([source[:N_TR, TMIN_PROBE - j : T - j] for j in frame_offsets], -1)
    Zte = np.concatenate([source[N_TR:N_PROBE, TMIN_PROBE - j : T - j] for j in frame_offsets], -1)
    d = Ztr.shape[-1]
    Ztr, Zte = Ztr.reshape(-1, d).astype(np.float64), Zte.reshape(-1, d).astype(np.float64)
    B = lstsq_fit(PVtr, Ztr)
    Ztr, Zte = Ztr - lstsq_apply(B, PVtr), Zte - lstsq_apply(B, PVte)
    r2 = lambda p: float(1 - ((p - res_te) ** 2).sum() / (res_te**2).sum())  # noqa: E731
    real = r2(lstsq_apply(lstsq_fit(Ztr, res_tr), Zte))
    shuf = r2(lstsq_apply(lstsq_fit(Ztr[rng.permutation(len(Ztr))], res_tr), Zte))
    return dict(label=label, d=d, real=real, shuf=shuf, net=real - shuf)


tests = [
    explain_residual(pos_t, [1], "past positions, 1 frame"),
    explain_residual(pos_t, range(1, 3), "past positions, 2 frames"),
    explain_residual(pos_t, range(1, 6), "past positions, 5 frames"),
    explain_residual(pos_t, range(1, 11), "past positions, 10 frames"),
    explain_residual(test.obs[:N_PROBE], [0], "**present** observation obs(t) only"),
    explain_residual(test.obs[:N_PROBE], [1], "past observations, obs(t−1) only"),
    explain_residual(test.obs[:N_PROBE], [2], "past observations, obs(t−2) only"),
    explain_residual(test.obs[:N_PROBE], [5], "past observations, obs(t−5) only"),
    explain_residual(test.obs[:N_PROBE], range(1, 3), "past observations, 2 frames"),
    explain_residual(test.obs[:N_PROBE], range(1, 6), "past observations, 5 frames"),
    explain_residual(test.obs[:N_PROBE], range(1, 11), "past observations, 10 frames"),
    explain_residual(test.obs[:N_PROBE], range(0, 5), "present + past observations, 5 frames"),
]
rows = ["| predictor set (all residualised vs present (pos,vel)) | # predictors | held-out R² | shuffled control | net |",
        "|---|---|---|---|---|"]
for t_ in tests:
    rows.append(f"| {t_['label']} | {t_['d']} | **{t_['real']:.4f}** | {t_['shuf']:+.4f} | {t_['net']:+.4f} |")
display(Markdown(
    f"**Table 2 — what explains the fiber residual** (linear fiber residual = {fiber_lin:.4f} of ‖h‖, "
    f"MLP = {fiber_mlp:.4f}). Held-out R² of the residual, not of `h`.\n\n" + "\n".join(rows)))

In [ ]:
# [7] Fig 2 — horizontal bars (long labels), so what explains the complement is readable at a glance
fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.6))
for ax in axes:
    style_ax(ax)

ax = axes[0]
labels = [t_["label"].replace("**", "") for t_ in tests]
vals = [t_["real"] for t_ in tests]
cols = [OI["orange"] if lb.startswith("past positions")
        else (OI["green"] if lb.startswith("present") else OI["blue"]) for lb in labels]
y = np.arange(len(labels))
ax.barh(y, vals, color=cols, height=0.72)
ax.barh(y, [t_["shuf"] for t_ in tests], color=OI["red"], height=0.28, label="shuffled control")
for i, lb in enumerate(labels):
    if lb.startswith("past positions"):
        ax.text(0.012, i, "≈ 0", va="center", ha="left", fontsize=7.5, color=OI["orange"])
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("held-out R² of the fiber residual")
ax.set_xlim(-0.07, 0.8)
ax.axvline(0, color=OI["grey"], lw=1)
ax.set_title("(a) past positions explain nothing; observations explain most of it", fontsize=10)
ax.legend(fontsize=8, loc="center right", bbox_to_anchor=(1.0, 0.30))

ax = axes[1]
single = [(0, "obs(t)"), (1, "obs(t−1)"), (2, "obs(t−2)"), (5, "obs(t−5)")]
sv = [explain_residual(test.obs[:N_PROBE], [j], lb)["real"] for j, lb in single]
ax.plot([j for j, _ in single], sv, "o-", color=OI["blue"], lw=2, ms=6,
        label="a single observation frame, at offset j")
ax.plot([1, 2, 5, 10],
        [tests[i]["real"] for i in (5, 8, 9, 10)], "s--", color=OI["sky"], lw=2, ms=5,
        label="cumulative: obs(t−1) … obs(t−j)")
ax.axhline(tests[0]["real"], color=OI["orange"], lw=2, ls=":", label="past positions (any number of frames)")
ax.axhline(0, color=OI["red"], lw=1, ls=":", label="shuffled control")
ax.set_xlabel("frame offset j into the past")
ax.set_ylabel("held-out R² of the fiber residual")
ax.set_title("(b) the trace decays with age — a recent-observation memory", fontsize=10)
ax.legend(fontsize=8, loc="center right")
ax.set_ylim(-0.05, 0.8)

fig.suptitle("Fig 2 — what predicts the part of h that is NOT a function of the present (pos, vel) "
             "(GRU · 256 hidden · dataset 4, held-out sequences)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## §3 — Does writing the whole history fix the edit?

The editor the hypothesis implies: instead of writing one frame's position into `h`, write the **entire
believed history**, rigidly translated by the teleport displacement `δ`. Same `δ` at every lag preserves
velocity, so the past the model now believes in is the one that *arrives at the target at the right speed*.

Note what this is algebraically. In a world where velocity is constant, `A_k ≈ A_pos − k·dt·A_vel`, so
demanding `A_k Δh = δ` for every `k` is equivalent to **`A_pos Δh = δ` and `A_vel Δh = 0`** — a
*velocity-pinned* position injection. That is still a new editor (plain injection lets the velocity readout
move as a min-norm side effect), but it predicts the `n`-sweep will **saturate** rather than grow, and cell
[9] measures the effective rank to check.

**The comparison that makes this decisive:** the identical content is also delivered through the
**observation channel** — the model is teacher-forced on `n+1` *rendered* frames of the same translated
world. Same displacement, same frames, same `n`. Only the channel differs.

### Pre-registered interpretation (fixed before running — `research/directions/history-editing.md`)

| outcome | reading |
|---|---|
| Latent history translation ≫ Pseudoinverse Injection, fidelity ≤ 1 | **Edit completeness was the barrier.** Latent editing re-opens. |
| Latent ≈ Pseudoinverse Injection while the observation twin lands | The **channel** is the barrier, not the content. Hardens the thread's central negative. |
| Latent moves only with fidelity > 1, or matches the matched-norm random control | Degradation / write-size, not editing. |

In [ ]:
# [8] the edit setup: the two ground-truth worlds, the displacement delta, the pre-edit state
NE = N_EDIT
edit_obj = edits.edit_object[:NE].astype(int)
sample_ix = np.arange(NE)
obs_e = edits.obs[:NE]
pos_e = edits.positions[:NE, :, :N_OBJ, :].astype(np.float32)
with h5py.File(edits.h5_path, "r") as f:
    vel_e = f["velocities"][:NE, :, :N_OBJ, :].astype(np.float32)

gt_roll = edits.clean_obs[:NE, EF : EF + K].astype(np.float32)   # the CLEAN post-edit world
zones = build_edit_zones(
    pre_pos=pos_e[:, EF - 1], tgt_pos=pos_e[:, EF], pre_vel=vel_e[:, EF - 1],
    edit_object=edit_obj, sim=sim, n_obj=N_OBJ,
    traj_pos=pos_e[:, EF : EF + K], gt_edited_traj=gt_roll,
)

# delta: target minus where the object WOULD have been at ef. Adding it at every lag translates
# the past rigidly, which leaves the velocity untouched (the fixed-velocity assumption).
would_be = pos_e[:, EF - 1] + vel_e[:, EF - 1] * dt
delta = np.zeros((NE, N_OBJ, 2), np.float32)
delta[sample_ix, edit_obj] = pos_e[sample_ix, EF, edit_obj] - would_be[sample_ix, edit_obj]

h0 = warm_up_to(obs_e, EF)                     # teacher-forced obs[0..ef-1]; encodes frame ef-1
h0n = h0.cpu().numpy().astype(np.float64)
print(f"N = {NE} held-out edits | mean ‖δ‖ = {np.linalg.norm(delta[sample_ix, edit_obj], axis=-1).mean():.3f} "
      f"sim units | mean teleport = {zones.teleport.mean():.3f}")
print(f"zones: {zones.target.sum(1).mean():.1f} target rays, {zones.ghost.sum(1).mean():.1f} ghost rays, "
      f"{zones.differing.sum(1).mean():.1f} differing rays (of {R})")

# probe bank: states from the edits split. Rows are frames TMIN_EDIT..ef-1 — pre-edit, and the
# SAME rows for every n, so only the number of lag blocks changes across the sweep.
hb = hidden_states(edits.obs[:N_BANK])
pos_b = edits.positions[:N_BANK, :, :N_OBJ, :].reshape(N_BANK, -1, 4).astype(np.float64)
rows_t = np.arange(TMIN_EDIT, EF)
print(f"editor probe bank: {N_BANK} sequences x frames {rows_t[0]}..{rows_t[-1]} = "
      f"{N_BANK * len(rows_t)} rows, identical for every n")


def stacked_probe(n):
    """A_n : h -> [pos(t), pos(t-1), ..., pos(t-n)], one least-squares solve."""
    X = hb[:N_BANK, rows_t].reshape(-1, H).astype(np.float64)
    Y = np.concatenate([pos_b[:, rows_t - k] for k in range(n + 1)], -1).reshape(-1, 4 * (n + 1))
    A = lstsq_fit(X, Y)
    return A[:-1].T, A[-1]

In [ ]:
# [9] Fig 3 — the stacked probe's spectrum: how many dimensions does "the history" actually add?
spectra = {n: np.linalg.svd(stacked_probe(n)[0], compute_uv=False) for n in NS}
rank_rows = ["| n (past frames written) | probe outputs 4(n+1) | numeric rank (1e-6) | **effective rank (1%)** | "
             "singular values 1–6 |", "|---|---|---|---|---|"]
for n in NS:
    sv = spectra[n]
    rank_rows.append(
        f"| {n} | {4*(n+1)} | {int((sv > sv[0]*1e-6).sum())} | **{effective_rank(sv)}** | "
        f"{np.array2string(sv[:6], precision=2, separator=', ')} |")
display(Markdown("**Table 3 — the stacked lag probe's rank.** The numeric rank grows as 4(n+1); the rank that "
                 "carries energy does not.\n\n" + "\n".join(rank_rows)))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
for ax in axes:
    style_ax(ax)
ax = axes[0]
for n, c in zip(NS, [OI["grey"], OI["sky"], OI["blue"], OI["green"], OI["orange"], OI["red"]]):
    sv = spectra[n]
    ax.semilogy(np.arange(1, len(sv) + 1), sv / sv[0], "o-", ms=3, lw=1.5, color=c,
                label=f"n = {n}  ({4*(n+1)} outputs)")
ax.axhline(1e-2, color="k", lw=1, ls="--", label="1% of the largest (effective-rank cut)")
ax.set_xlabel("singular value index")
ax.set_ylabel("singular value / largest")
ax.set_xlim(0, 40)
ax.set_title("(a) spectrum — a 4-strong core, then a cliff", fontsize=10)
ax.legend(fontsize=7.5)

ax = axes[1]
ax.plot(NS, [4 * (n + 1) for n in NS], "s--", color=OI["grey"], lw=2, ms=6,
        label="probe outputs 4(n+1) — what the write LOOKS like")
ax.plot(NS, [effective_rank(spectra[n]) for n in NS], "o-", color=OI["blue"], lw=2, ms=7,
        label="effective rank — what the write can actually use")
ax.axhline(8, color=OI["orange"], lw=1.5, ls=":", label="8 = the (pos, vel) core")
ax.set_xlabel("n (past frames written)")
ax.set_ylabel("dimensions")
ax.set_title("(b) adding history adds outputs, not usable dimensions", fontsize=10)
ax.legend(fontsize=8, loc="upper left")

fig.suptitle("Fig 3 — rank of the stacked lag probe A_n : h → [pos(t) … pos(t−n)] "
             "(GRU · 256 hidden · dataset 4)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
# [10] build every arm: the same translated history through the LATENT and through OBSERVATIONS
ARMS = {"Unsteered": h0n}

# --- reference: the thread's published readout injection (probe on all frames, target positions[ef])
Xa = hb.reshape(-1, H).astype(np.float64)
Aa = lstsq_fit(Xa, pos_b[:, : hb.shape[1]].reshape(-1, 4))
W_pub, b_pub = Aa[:-1].T, Aa[-1]
tgt_flat = pos_e[:, EF].reshape(NE, 4).astype(np.float64)
ARMS["Pseudoinverse Injection (published)"] = (
    h0n + (tgt_flat - (h0n @ W_pub.T + b_pub)) @ np.linalg.pinv(W_pub).T
)

# --- LATENT channel: rigid translation of the believed history
for n in NS:
    W, b_ = stacked_probe(n)
    W_pinv = np.linalg.pinv(W)
    target = np.concatenate(
        [(pos_e[:, EF - 1 - k] + delta).reshape(NE, 4) for k in range(n + 1)], -1
    ).astype(np.float64)
    current = h0n @ W.T + b_
    ARMS[f"Latent history translation · n={n}"] = h0n + (target - current) @ W_pinv.T
    if n == 8:  # control: same probe, but only the CURRENT frame is displaced
        target_i = np.concatenate(
            [(pos_e[:, EF - 1 - k] + (delta if k == 0 else 0)).reshape(NE, 4) for k in range(n + 1)], -1
        ).astype(np.float64)
        ARMS["Latent, inconsistent history · n=8"] = h0n + (target_i - current) @ W_pinv.T

# --- OBSERVATION channel: the identical content, rendered and teacher-forced
render_cfg = SimConfig(
    seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
    n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=dt, obs_res=R,
    refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
    obs_noise_std=0.0, boundary="open", always_in_frustum=False,
)
refl = np.linspace(sim["refl_min"], sim["refl_max"], N_OBJ).astype(np.float32)
radii = np.full(N_OBJ, sim["radius"], np.float32)

hist_frames = np.zeros((NE, max(NS) + 1, R), np.float32)   # index j = frame ef-1-j, translated world
for j in range(max(NS) + 1):
    p = pos_e[:, EF - 1 - j] + delta
    for i in range(NE):
        _, _, inten = render_frame(p[i], radii, refl, render_cfg)
        hist_frames[i, j] = inten

for n in NS:
    h_start = warm_up_to(obs_e, EF - 1 - n)            # real (noisy) observations up to ef-2-n
    frames = hist_frames[:, : n + 1][:, ::-1]          # oldest first: ef-1-n … ef-1
    ARMS[f"Observation history overwrite · n={n}"] = teacher_force(h_start, frames).cpu().numpy()

# --- controls on the write itself
d_best = ARMS["Latent history translation · n=8"] - h0n
rng2 = np.random.default_rng(0)
rand_dir = rng2.standard_normal(d_best.shape)
rand_dir *= np.linalg.norm(d_best, axis=1, keepdims=True) / np.linalg.norm(rand_dir, axis=1, keepdims=True)
ARMS["Random direction, matched norm (n=8)"] = h0n + rand_dir
for a in (2.0, 4.0):
    ARMS[f"Latent history translation · n=8, scaled x{a:g}"] = h0n + a * d_best

ROLL = {k: rollout(v) for k, v in ARMS.items()}
print(f"{len(ARMS)} arms rolled out for K={K} steps each")

In [ ]:
# [11] the canonical §4 scorecard for every arm
cards = {k: edit_scorecard(v, zones, gt_roll) for k, v in ROLL.items()}
for k, c in cards.items():
    c["fidelity_ratio"] = fidelity_ratio(c, cards["Unsteered"])
    c["dh"] = float(np.linalg.norm(ARMS[k] - h0n, axis=1).mean() / np.linalg.norm(h0n, axis=1).mean())

order = (["Unsteered", "Pseudoinverse Injection (published)"]
         + [f"Latent history translation · n={n}" for n in NS]
         + ["Latent, inconsistent history · n=8", "Random direction, matched norm (n=8)"]
         + [f"Latent history translation · n=8, scaled x{a:g}" for a in (2.0, 4.0)]
         + [f"Observation history overwrite · n={n}" for n in NS])
rows = ["| arm | channel | **Edit Index** ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | "
        "GT-traj RMSE ↓ | **fidelity ratio** ↓ | ‖Δh‖/‖h‖ |", "|---|---|---|---|---|---|---|---|---|"]
for k in order:
    c = cards[k]
    ch = ("observation" if "Observation" in k else "—" if k == "Unsteered" else "latent")
    flag = " ⚠" if c["fidelity_ratio"] > 1.05 else ""
    rows.append(
        f"| {k} | {ch} | **{c['edit_index']:+.3f}** | {c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | "
        f"{c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f}{flag} | "
        f"{c['dh']:.3f} |")
display(Markdown(
    "**Table 4 — the canonical §4 scorecard.** ⚠ marks fidelity ratio > 1.05: the edited rollout ended "
    "*further* from the true post-edit world than doing nothing, so any Edit Index movement there is "
    "degradation, not steering.\n\n" + "\n".join(rows)))

print(f"\nheadline: latent n=8 {cards['Latent history translation · n=8']['edit_index']:+.3f} "
      f"vs matched-norm RANDOM {cards['Random direction, matched norm (n=8)']['edit_index']:+.3f} "
      f"vs observation n=8 {cards['Observation history overwrite · n=8']['edit_index']:+.3f} "
      f"(unsteered {cards['Unsteered']['edit_index']:+.3f})")

In [ ]:
# [12] Fig 4 — same content, same n, two channels
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for ax in axes:
    style_ax(ax)

lat = [cards[f"Latent history translation · n={n}"]["edit_index"] for n in NS]
obs_ = [cards[f"Observation history overwrite · n={n}"]["edit_index"] for n in NS]

ax = axes[0]
ax.plot(NS, obs_, "o-", color=OI["green"], lw=2.4, ms=7, label="observation channel (rendered frames)")
ax.plot(NS, lat, "s-", color=OI["blue"], lw=2.4, ms=7, label="latent channel (write to h)")
ax.axhline(cards["Unsteered"]["edit_index"], color=OI["grey"], lw=1.6, ls="--", label="unsteered")
ax.axhline(cards["Random direction, matched norm (n=8)"]["edit_index"], color=OI["red"], lw=1.6, ls=":",
           label="random direction, matched norm (n=8)")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("n = past frames written (history depth)")
ax.set_ylabel("Edit Index  (+1 = edited world, −1 = unedited)")
ax.set_title("(a) identical content, identical n — only the channel differs", fontsize=10)
ax.legend(fontsize=7.5, loc="center right")
ax.set_ylim(-0.8, 0.8)

ax = axes[1]
ax.plot(NS, [cards[f"Observation history overwrite · n={n}"]["fidelity_ratio"] for n in NS], "o-",
        color=OI["green"], lw=2.4, ms=7, label="observation channel")
ax.plot(NS, [cards[f"Latent history translation · n={n}"]["fidelity_ratio"] for n in NS], "s-",
        color=OI["blue"], lw=2.4, ms=7, label="latent channel")
ax.axhline(1.0, color=OI["red"], lw=1.6, ls="--", label="1.0 — above this the edit DEGRADES")
ax.set_xlabel("n = past frames written (history depth)")
ax.set_ylabel("fidelity ratio (GT-traj RMSE / unsteered)")
ax.set_title("(b) fidelity — the observation arm genuinely improves the rollout", fontsize=10)
ax.legend(fontsize=8)

ax = axes[2]
bar_arms = ["Unsteered", "Pseudoinverse Injection (published)", "Latent history translation · n=8",
            "Random direction, matched norm (n=8)", "Latent, inconsistent history · n=8",
            "Latent history translation · n=8, scaled x4", "Observation history overwrite · n=8"]
short = ["Unsteered", "Pseudoinverse Injection\n(published, latent)", "Latent history\ntranslation n=8",
         "Random direction,\nmatched norm (n=8)", "Latent, inconsistent\nhistory n=8",
         "Latent history transl.\nn=8, scaled x4", "Observation history\noverwrite n=8"]
vals = [cards[a]["edit_index"] for a in bar_arms]
fid = [cards[a]["fidelity_ratio"] for a in bar_arms]
colors = [OI["grey"], OI["sky"], OI["blue"], OI["red"], OI["purple"], OI["orange"], OI["green"]]
yy = np.arange(len(bar_arms))
ax.barh(yy, vals, color=colors, height=0.7)
for i, (v, f_) in enumerate(zip(vals, fid)):
    # annotate just PAST the zero line on the bar's own side, so the text never lands on the
    # y-tick labels (which sit to the left of the axis)
    ax.text(v + 0.03 if v >= 0 else 0.03, i, f"fid {f_:.2f}", va="center", ha="left", fontsize=7.5,
            color=OI["red"] if f_ > 1.05 else OI["grey"])
ax.set_yticks(yy)
ax.set_yticklabels(short, fontsize=7.5)
ax.invert_yaxis()
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Edit Index")
ax.set_xlim(-0.85, 0.95)
ax.set_title("(c) the controls — direction vs write size", fontsize=10)

fig.suptitle("Fig 4 — rigidly translating the believed history: through the latent vs through observations "
             "(GRU · 256 hidden · dataset 4, N=256 held-out edits)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## §4 — Where does a *successful* edit live, relative to the history probe?

The observation-channel arm gives a working edit and therefore a ground-truth edit direction
`Δh_true = h(observation overwrite, n=16) − h₀`. Any injection-style editor writing through `A_n` is
confined to `row(A_n)` **by construction**, so the fraction of `Δh_true` lying in that row space is the
**hard ceiling** on what a latent history write could ever achieve.

The rank of `A_n` changes across the sweep, so the chance level `√(rank/H)` moves with it. Per `CLAUDE.md`,
the **enrichment** (`fraction / chance`) is the quantity plotted — the raw fraction would manufacture a
trend that is entirely the moving chance level. Ranks are the **effective** ranks from Fig 3.

In [ ]:
# [13] Fig 5 — the reachability ceiling: is the true edit any more available as n grows?
dh_true = ARMS[f"Observation history overwrite · n={max(NS)}"] - h0n
geo = {}
for n in NS:
    W, _ = stacked_probe(n)
    _, sv, Vt = np.linalg.svd(W, full_matrices=False)
    r_eff = effective_rank(sv)
    frac = float(subspace_fraction(dh_true, Vt[:r_eff]).mean())
    ch = chance_fraction(r_eff, H)
    geo[n] = dict(rank=r_eff, frac=frac, chance=ch, enrich=frac / ch)

grows = ["| n | effective rank | ‖P·Δh_true‖/‖Δh_true‖ | chance √(rank/H) | **enrichment** |",
         "|---|---|---|---|---|"]
for n in NS:
    g = geo[n]
    grows.append(f"| {n} | {g['rank']} | {g['frac']:.4f} | {g['chance']:.4f} | **{g['enrich']:.2f}×** |")
display(Markdown("**Table 5 — reachability ceiling of the latent history write.** `Δh_true` is the state "
                 "change produced by the *working* observation-channel edit at n=16.\n\n" + "\n".join(grows)))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
for ax in axes:
    style_ax(ax)
ax = axes[0]
ax.plot(NS, [geo[n]["frac"] for n in NS], "o-", color=OI["blue"], lw=2.2, ms=7,
        label="fraction of Δh_true inside row(A_n)")
ax.plot(NS, [geo[n]["chance"] for n in NS], "s--", color=OI["grey"], lw=2, ms=6,
        label="chance level √(rank/H) — it MOVES with n")
ax.set_xlabel("n (past frames written)")
ax.set_ylabel("fraction of ‖Δh_true‖")
ax.set_title("(a) raw fraction — uninterpretable without its chance level", fontsize=10)
ax.legend(fontsize=8)
ax.set_ylim(0, 0.25)

ax = axes[1]
ax.plot(NS, [geo[n]["enrich"] for n in NS], "o-", color=OI["blue"], lw=2.4, ms=8)
ax.axhline(1.0, color=OI["red"], lw=1.8, ls="--", label="1.0 = chance (a random direction does as well)")
ax.set_xlabel("n (past frames written)")
ax.set_ylabel("enrichment  (fraction / chance)")
ax.set_title("(b) enrichment — at or below chance for every history depth", fontsize=10)
ax.legend(fontsize=8)
ax.set_ylim(0, 1.4)

fig.suptitle("Fig 5 — how much of a WORKING edit is reachable through the history probe's row space "
             "(GRU · 256 hidden · dataset 4, N=256)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## §5 — Observation space: what the two channels actually generate

A scorecard compresses a rollout to one number and routinely hides the difference between *the edit landed*
and *the output degraded* — the two look identical in an Edit Index that moved. Required by `CLAUDE.md` for
any claim about an effect on generations, and built through the single `waterfall_grid(...)` helper in
`history_tools.py` (gray on dark; a GT column of clean sim observations; six **noisy** pre-edit context
frames above the dashed edit line; below it, **every column is its own free-run from step 0**; solid green =
target, dashed red = ghost).

Each column title carries that arm's Edit Index and fidelity ratio, so the picture and the number are read
together. The degenerate settings (`inconsistent history`, `scaled ×4`) appear as their own columns — that
is where collapse shows up.

In [ ]:
# [14] Fig 6 — the mandatory observation-space waterfall, through the ONE spec helper
wf_arms = ["Unsteered", "Pseudoinverse Injection (published)", "Latent history translation · n=0",
           "Latent history translation · n=8", "Latent, inconsistent history · n=8",
           "Latent history translation · n=8, scaled x4", "Observation history overwrite · n=0",
           "Observation history overwrite · n=8"]
wf_short = {
    "Unsteered": "Unsteered",
    "Pseudoinverse Injection (published)": "Pseudoinverse Injection\n(published, latent)",
    "Latent history translation · n=0": "Latent history\ntranslation n=0",
    "Latent history translation · n=8": "Latent history\ntranslation n=8",
    "Latent, inconsistent history · n=8": "Latent, inconsistent\nhistory n=8",
    "Latent history translation · n=8, scaled x4": "Latent history transl.\nn=8, scaled x4",
    "Observation history overwrite · n=0": "Observation history\noverwrite n=0",
    "Observation history overwrite · n=8": "Observation history\noverwrite n=8",
}
labels = {a: f"{wf_short[a]}\nEdit Index {cards[a]['edit_index']:+.2f} · fid {cards[a]['fidelity_ratio']:.2f}"
          for a in wf_arms}

# show the largest teleports — the samples where an edit is most visible either way
samples = list(np.argsort(zones.teleport)[::-1][:3])
fig = waterfall_grid(
    rolls={a: ROLL[a] for a in wf_arms},
    ctx=edits.obs[:NE, EF - 6 : EF].astype(np.float32),
    gt_roll=gt_roll,
    tgt_cx=ray_centroid(zones.target),
    ghost_cx=ray_centroid(zones.ghost),
    samples=samples,
    edit_frame=EF,
    leads_by_one=(),
    title=("Fig 6 — observation-space rollouts after rigidly translating the believed history, "
           "latent channel vs observation channel\n(GRU · 256 hidden · dataset 4; the three largest "
           "teleports; each column is that arm's own free-run)"),
    labels=labels,
)
out_dir = Path("/tmp/history_editing")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "fig6_gru_waterfall.png", dpi=125, bbox_inches="tight", facecolor="#0a0a14")
plt.show()
print("saved", out_dir / "fig6_gru_waterfall.png")

## Summary

**What this notebook measures** (invariant): whether the part of the GRU's hidden state that latent edits
never touch is *the past*, and whether writing the whole believed history — rather than one frame — makes
the edit land. Definitions and formulas are above; the numbers below move as models and experiments evolve.

### Current results (updated 2026-08-13) — GRU · 256 hidden · dataset 4

**1. The past is readable from `h`, and only nonlinearly.** A *linear* probe reads `pos(t−k)` about as well
at every lag (R² 0.828 at k=0, 0.834 at k=5, 0.784 at k=20) — but so does the no-stored-history null, to
within **+0.0008 R² at every lag**. Linearly, `h` carries no history at all beyond `(pos, vel)`.
The **MLP** probe separates: direct **0.883** vs the learned null **0.737** at k=20, a **+0.146** gain that
grows monotonically with lag (+0.002 at k=2, +0.042 at k=10, +0.089 at k=15). The imposed and learned nulls
agree to ≤0.006, so the gap is **not** extrapolation inefficiency. `h` does store information about the past
that its own `(pos, vel)` readout does not imply — nonlinearly, exactly as velocity is stored.
*Calibration:* the direct MLP probe still sits **below** the true-`(pos, v)` ceiling (0.991), so `h`'s
knowledge of the past does not exceed what perfect knowledge of the present would imply.

**2. The un-edited complement is observation content, not past positions.** The linear fiber residual is
**0.856** of ‖h‖ (MLP **0.467**) — that is the "extra information" the hypothesis concerns. Regressed on
predictors that have first been residualised against the present `(pos, vel)`:

* **past positions explain ~0%** of it — held-out R² −0.0001 to −0.0007, indistinguishable from the
  shuffled control, at 1, 2, 5 or 10 frames.
* **observations explain most of it** — obs(t) alone **0.659**, obs(t−1) alone **0.609**, obs(t−2) **0.550**,
  obs(t−5) **0.364**, decaying with age; ten past frames reach **0.636** against a shuffled control of −0.060.

So the complement is a **decaying trace of recent sensory frames** — silhouette detail, noise realisations,
whatever `(pos, vel)` throws away — and *not* a record of where the objects were.

**3. Writing the translated history into `h` does not edit the world; writing it through observations does.**
With the content, the displacement `δ` and the number of frames `n` held fixed and **only the channel varied**:

| n (past frames written) | latent channel | observation channel |
|---|---|---|
| 0 | −0.665 | +0.028 |
| 1 | −0.663 | +0.300 |
| 2 | −0.657 | +0.432 |
| 4 | −0.634 | +0.554 |
| 8 | −0.585 | **+0.635** |
| 16 | −0.477 ⚠ fid 1.06 | **+0.671** (fid 0.61) |

unsteered −0.670. **The decisive control: a *random* direction of matched norm scores −0.585 — exactly the
latent n=8 value.** Every point of the latent arm's apparent gain is bought by the size of the write, not by
its content; and past n=8 the gain only continues by degrading (fidelity 1.06 at n=16, 1.36 when scaled ×4).
The observation arm reaches the thread's published counterfactual-overwrite ceiling (+0.68) at fidelity 0.61,
with Target RMSE 0.488 → 0.104 and Ghost 0.589 → 0.107 while collateral stays flat.

**4. Why the latent write cannot work, structurally.** The stacked lag probe's **effective rank saturates at
8** — the `(pos, vel)` core — however many lag blocks are stacked on (numeric rank 4, 8, 12, 20, 36, 68;
effective rank 4, 7, 8, 8, 8, 9). "Writing the history" adds probe *outputs*, not usable *dimensions*. And
`Δh_true` from the working observation edit lies in `row(A_n)` at **0.49–0.60× chance** for every `n` — at or
**below** chance, and *falling* as history is added. The demand for an *inconsistent* history (δ at lag 0
only) forces the near-null trailing directions and produces a write of **5.5× ‖h‖**, which collapses the
rollout into vertical-stripe garbage (fidelity 2.77, Fig 6).

### Interpretation (mine, not established)

The hypothesis is **half right, and the half that is right explains the negative rather than lifting it.**
The un-edited complement *is* history — but it is **observation-shaped** history, not a decodable record of
past positions. That resolves the pre-registered fork onto its second branch: **the channel is the barrier,
not the content.** A position probe's row space is an 8-dimensional `(pos, vel)` object, and no amount of
stacking lags enlarges it, so a probe-derived write cannot reach observation-shaped content *in principle* —
whereas teacher-forcing rendered frames writes exactly that content and lands the edit.

This gives the thread's through-line — *no successful edit is free of dynamics* — a mechanism: the working
editors are not merely "using the dynamics", they are the only ones writing in the **format the complement is
stored in**. It also predicts what the companion transformer notebook tests: where the history is the carried
state, with a per-frame slot per token, the same content may be writable in activation space.

### Owed / scope limits

* One model (`controls/H256`), one seed, one dataset, position only; N=256 edits, N=1500 probe sequences.
* §2's residual decomposition is **linear**; a nonlinear map from past frames to the residual would likely
  explain more, and the MLP fiber residual (0.467) is the tighter target it should be run against.
* The observation arm is fed **clean** renders while the model was trained on noisy observations — a mild
  optimistic bias for the channel that already wins, so it does not threaten the conclusion, but a
  noise-matched version is the obvious tightening.
* The `n`-sweep saturation is *predicted* by the ballistic world (constant velocity). A world with
  acceleration or bounces would make past positions genuinely independent of `(pos, vel)`, and is the
  natural follow-on if one wants the literal form of the hypothesis tested where it could win.